# <span style="color:orange"> *Multiple bounded optimization* </span> 

In [1]:
%matplotlib inline
import nibabel as nib
import numpy as np
from scipy import optimize
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import subprocess
import sys
import pandas as pd
import os

 # <span style="color:lime"> *Repository & Tool loading* </span>
 Remember to check that it says the exact commit you should be working on </br>
 ### *This only needs to be run once, then the folder will be created*

Go to the command line and do this: </br>
!git clone https://github.com/shimming-toolbox/susceptibility-to-fieldmap-fft.git
%cd susceptibility-to-fieldmap-fft </br>
!git status </br>
%pip install . </br>

In [3]:
home_path = r"C:\Users\Admin\Documents\msc_project\Image-processing-strategies\chi_opt"
#Once we confirmed the head of the chi to fbfest we can go to home
%cd {home_path}
!ls

C:\Users\Admin\Documents\msc_project\Image-processing-strategies\chi_opt
__pycache__
chi_demostrator.py
chi_opt_debugging.ipynb
chi_opt_demod_32.ipynb
chi_opt_demod_32_35.ipynb
chi_opt_demod_33.ipynb
chi_opt_demod_35.ipynb
chi_opt_demod_x18.ipynb
chi_optimizer.py
demod.py
normalize_mprage.py
shazam_requirements.txt
susceptibility-to-fieldmap-fft
tissue-to-MRproperty
utils


In [5]:
path_to_chi_to_fm_fft = r"C:\Users\Admin\Documents\msc_project\Image-processing-strategies\chi_opt\susceptibility-to-fieldmap-fft"
sys.path.append(path_to_chi_to_fm_fft)
from functions import compute_fieldmap

 # <span style="color:silver"> *Data analysis* </span> 

In [5]:
# Define the run nfolder name
run_number = "test1_no_extra_fm_creation_test"

In [6]:
# Everytime you run the code, it will create a new folder with the run number and restart the counter
path_to_iter_fms = r"Z:\neuropoly_data\chi_fitting\fm\sim\chi_opt\iter_fms"
path_to_iter_metrics = r"Z:\neuropoly_data\chi_fitting\fm\sim\chi_opt\iter_metrics"

path_to_iter_fms = os.path.join(path_to_iter_fms, run_number)
path_to_iter_metrics = os.path.join(path_to_iter_metrics, run_number)
counter = 0

#########
# This don't change, this are used with the simulation's FOV
path_to_sim_metric_mask = r"Z:\neuropoly_data\chi_fitting\fm\sim\D2_D3_masks\t1w_wholebody_sc_msk_labeled.nii.gz" # Simulation
path_to_dmod_mask = r"Z:\neuropoly_data\chi_fitting\fm\sim\D2_D3_masks\t1w_wholebody_sc_msk.nii.gz" # Simulation
path_to_chimap = r"Z:\neuropoly_data\chi_fitting\fm\sim\B1_chi_maps\chi_018_chi_map.nii.gz" # -4.2 for both trachea and lungs as initial guess
path_to_segs = r"Z:\neuropoly_data\chi_fitting\fm\sim\final_segmentations.nii.gz"


##  <span style="color:#C133FF"> *Performing demodulation on measured!* </span> 

In [7]:
import importlib
import utils.demod   # your custom module (e.g., my_module.py)
importlib.reload(utils.demod)
from utils.demod import demod_Hz

In [8]:
# load in-vivo average metrics, this covers C3 to T8 (3 to 15)
invivo_avg_metrics = pd.read_csv(r"Z:\neuropoly_data\chi_fitting\fm\C_dmod_meas\simple_avg_respiration.csv")

In [9]:
# Check the values:
invivo_avg_metrics_values = invivo_avg_metrics['WA()']
invivo_avg_metrics_values
# Compare to the respective compare_fm participants notebook

0    -109.455607
1    -120.886076
2     -90.288441
3     -32.954174
4      44.038240
5     110.800425
6      96.833710
7      31.554322
8      27.002024
9      39.511551
10     36.282699
11     13.711603
12    -14.105621
Name: WA(), dtype: float64

In [10]:
# Get info from json files of the scanner, the central frequency should be the same for both EXP and INSP
central_freq_exp = 123.24935 # in MHz  123.24935 vs 123.248944 (Exp vs Insp)
gamma_bar = 42.58 # MHz/T
B0 = 3 # [T]
B0_used_scanner = central_freq_exp /gamma_bar
print("The B0 to use in the simulation should be: ", B0_used_scanner, "T")

The B0 to use in the simulation should be:  2.894536167214655 T


In [11]:
# For this scan F0 is different between Exp and Insp, what is the Tesla difference?
difference = 123.24935 - 123.248944 # In MHz/T
print("The difference in Tesla between Exp and Insp is: ", difference/gamma_bar, "T")

The difference in Tesla between Exp and Insp is:  9.534992954728534e-06 T


 # <span style="color:gold"> *Optimization loop!* </span> </br>
 With the graphs working as we expect them too, lets begin chi optimization

In [12]:
history = [] 
history_chi_trachea = []
history_chi_lungs = []

In [13]:
# Loading dependencies outside obj. function to decrease computational needs

# Load the simulated susceptibility map in ppm
sim_chi_img = nib.load(path_to_chimap)
sim_chi_data = sim_chi_img.get_fdata()

# Load segmentation labels that create the chimaps
ROI_img = nib.load(path_to_segs)
ROI_data = ROI_img.get_fdata()

# Find indices with the labels we want to update
ind_trachea = np.where((ROI_data == 113))
ind_lungs = np.where((ROI_data == 12))

dmod_sim_mask = nib.load(path_to_dmod_mask).get_fdata()
metric_sim_mask = nib.load(path_to_sim_metric_mask).get_fdata()


vertebra_label_map = {"C1": 1, "C2": 2, "C3": 3, "C4": 4, "C5": 5, "C6": 6, "C7": 7, "T1": 8, "T2": 9, "T3": 10, "T4": 11, "T5": 12, "T6": 13, "T7": 14, "T8": 15}
vertebrae_levels_opt =  ['C3', 'C4', 'C5', 'C6', 'C7', 'T1', 'T2', 'T3', 'T4', 'T5', 'T6', 'T7', 'T8'] # From 3 to 15

In [14]:
global opt_file_fn
opt_file_fn = os.path.join(path_to_iter_metrics, "optimization_log.txt")

In [15]:
def log_solution(counter, chi_trachea, chi_lungs, obj_val):
    # This function assumes you've previously defined a global variable best solution, as well as a opt filename.txt
    global best_solution
    if obj_val <= best_solution:
        if obj_val == best_solution:
            print("Found a solution with the same objective value, but different parameters.")
            with open(opt_file_fn, 'a') as file:
                file.write(f" Iteration #{counter}, Chi trachea: {chi_trachea} & Chi Lung: {chi_lungs}. Obj value: {best_solution} \n")
            return 0

        best_solution = obj_val
        
        with open(opt_file_fn, 'a') as file:
            file.write(f" \n")
            print(f"New best solution: Iteration #{counter}, Chi trachea: {chi_trachea} & Chi Lung: {chi_lungs}. Obj value: {best_solution} ")
        return 1
    
    else:
        print("No improvement in objective value.")
        return 0

In [16]:
def f_nomad_opt(x):
    global counter, best_solution
    counter += 1
    print('$$$$$$$$$$$$$$$$$$$$$$$$$')
    print(f"Iteration #{counter}")
    # COnver the PyNomad Eval Point to list for subscriptions
    chi_trachea =  x.get_coord(0)
    chi_lungs = x.get_coord(1)

    print(f"Chi for trachea: {chi_trachea}")
    print(f"Chi for lungs: {chi_lungs}")

    # Step 1. Update chi value of trachea and lungs
    sim_chi_data[ind_trachea] = chi_trachea
    sim_chi_data[ind_lungs] = chi_lungs
    
    # Step 2. Compute the FM
    # Load with function to get image_res
    chi_dist, image_res, affine_matrix = compute_fieldmap.load_sus_dist(path_to_chimap)

    sim_b0_ppm = compute_fieldmap.compute_bz(sim_chi_data, image_resolution = image_res)
    sim_b0_Hz = sim_b0_ppm * central_freq_exp

    chi1_name = str(str(float(f"{chi_trachea:.3f}")))#.replace(".","_") # to take away the minus sign can use .strip("-")) at the end
    chi2_name = str(str(float(f"{chi_lungs:.3f}")))#.replace(".","_")

    # Step 3. demodulate and extract metrics
    dmod_value = np.mean(sim_b0_Hz[dmod_sim_mask == 1])

    print(f"Demodulation value for this iteration: {dmod_value}")
    dmod_sim_Hz = sim_b0_Hz - dmod_value

    # Now extract metrics manually instead of with subprocess to make it faster

    dmod_sim_vert_values = []

    for v in vertebrae_levels_opt:
        level = vertebra_label_map[v]
        mask = (metric_sim_mask==level)
        mean_value = np.mean(dmod_sim_Hz[mask])
        dmod_sim_vert_values.append(mean_value)

    # Step 4. Compute objective value and log solution
    difference = np.linalg.norm(np.array(dmod_sim_vert_values) - np.array(invivo_avg_metrics_values))
    print(f"Objective value for this iteration: {difference}")
    history.append(difference)
    history_chi_trachea.append(chi_trachea)
    history_chi_lungs.append(chi_lungs)

    plot_sol = log_solution(counter, chi_trachea, chi_lungs, difference)

    # Step 5. If the objective value is lower than the best solution, save the FM and chi maps
    if plot_sol == 1:
        # Save the FM and chi maps
        fm_filename = f"sim_b0_dmod_chi_trachea_{chi1_name}_chilung_{chi2_name}.nii.gz"
        chi_filename = f"sim_chi_chi_trachea_{chi1_name}_chi_lung_{chi2_name}.nii.gz"

        fm_path = os.path.join(path_to_iter_fms, fm_filename)
        chi_path = os.path.join(path_to_iter_fms, chi_filename)

        # Save the demodulated FM
        nib.save(nib.Nifti1Image(dmod_sim_Hz, affine_matrix), fm_path)
        # Save the updated chi map
        nib.save(nib.Nifti1Image(sim_chi_data, affine_matrix), chi_path)

    else:
        print("No improvement, not saving.")

    
    rawBBO = str(difference)
    x.setBBO(rawBBO.encode("UTF-8"))
    return 1


 # <span style="color:#1FC7AB"> *Optimization results!* </span> </br>

In [17]:
# Before running please verify your outpaths:
print(path_to_iter_fms)
print(path_to_iter_metrics)

Z:\neuropoly_data\chi_fitting\fm\sim\chi_opt\iter_fms\test1_no_extra_fm_creation_test
Z:\neuropoly_data\chi_fitting\fm\sim\chi_opt\iter_metrics\test1_no_extra_fm_creation_test


In [18]:
# Reset counter if neceesary (alone verrsion)
# This should be done when defining new paths but sometimes can be overridden
# Use with care
counter

0

In [23]:
# Manually reset counter when errors interrupt the optimization run,
# Don't forget to delete the folders
counter = 0
counter

0

In [19]:
import PyNomad as nomad
import time

In [20]:
# Using PyNomad for optimization# Using optimize minimize from scipy
# Set initial values, boundaries and run optimization
nomad_params = [
    "DIMENSION 2", 
    "BB_INPUT_TYPE (R R)",
    "BB_OUTPUT_TYPE OBJ",
    "MAX_BB_EVAL 3",
    "DISPLAY_DEGREE 2",
    "DISPLAY_ALL_EVAL false",
    "DISPLAY_STATS BBE OBJ",
    "VNS_MADS_SEARCH true", # Optional Variable Neighborhood Search
    "VNS_MADS_SEARCH_TRIGGER 0.75" # Max desired ration of VNS BBevals over the total number of BBevals
]
x0 = [0.3, -4.2] # 
# First bound is trachea // Depends on objective code !!!
# Second bound is Lung // Depends on objective code !!!
# Check the MD above!
lb = [-5, -5]
ub = [0.27, 0.2]

if counter != 0 :
        # This means that you forgot to change the folder run number, to avoid mixing tests, please run that cell 
        # Changing the number after run!
    print("Please change run # to avoid mixing result folders :)")
else:
    start_time = time.time()
    result = nomad.optimize(f_nomad_opt, x0, lb, ub, nomad_params)
    fmt = ["{} = {}".format(n,v) for (n,v) in result.items()]
    output = "\n".join(fmt)
    print("\nNOMAD results \n" + output + " \n")

    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"Optimization complete in: {elapsed_time:.3f} seconds")

: 

In [ ]:
chi_values_range = np.linspace(-1, 0.5, 100)

plt.figure(figsize=(8, 5))
plt.plot(history, marker='o', linestyle='-')
plt.xlabel("Iteration")
plt.ylabel("Objective Function Value")
plt.title("Convergence Plot 1D for Chi Air")
plt.grid()
plt.show()

In [ ]:
plt.figure()
chi_values_range = np.linspace(-1, 0.5, 100)
plt.plot(history_chi_lungs, marker='2', linestyle='--')
plt.plot(history_chi_trachea, marker='4', linestyle='-.')

plt.xlabel("Iteration")
plt.ylabel("Objective Function Value")
plt.title("Convergence pf Chi Trachea & Lungs")
plt.legend(["Lungs", "Trachea"])
plt.grid()
plt.show()

 # <span style="color:#D8C40D6A"> *Metrics!* </span> </br>
 Let's analyze how much we've improved after optimization

In [ ]:
worst_l2 = np.max(history)
print("Worst L2 norm: ", worst_l2)
best_l2 = np.min(history)
print("Best L2 norm: ", best_l2)

Worst L2 norm:  nan
Best L2 norm:  nan
